# 05 — Exp 4 & Exp 5: 7-Point Systematic Ablations (`A1`–`A7`) & FIR–MIR Pareto Scaling

Reuses the persistent Google Drive `api_cache.sqlite` so shared Stage-1 calls are never billed twice.
- **Ablations `A1`–`A7`**: w/o Jev Triage, OpenAI Luna Triage, Asymmetric Evidence (`A3`), Gate Decomposition (`A4`), Binary Only (`A5`), w/o $U_t$ Disagreement (`A6`), w/o Skill Routing (`A7`).
- **FIR–MIR Pareto Sweep**: $\lambda \in \{0.5, 0.75, 1.0, 1.25, 1.5, 2.0\}$ zero-cost rescoring.
- **Model Scaling**: `Luna` vs `Terra`.

In [ ]:
# === CELL 0: Mount Google Drive & Configure Persistent Workspace ===
import os
from pathlib import Path

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/when2speak_research")
else:
    DRIVE_ROOT = Path("./local_drive_mirror").resolve()

for sub in [
    "secrets",
    "config_cache",
    "datasets/pagemantle",
    "datasets/when2speak",
    "datasets/pagemantle_ext",
    "caches",
    "runs",
    "paper_artifacts/tables",
    "paper_artifacts/figures",
]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

os.environ["DIAL_SKILL_DRIVE_ROOT"] = str(DRIVE_ROOT)
print("Persistent DRIVE_ROOT:", DRIVE_ROOT)


In [ ]:
# === CELL 1: Sync GitHub Repo (git clone / git pull) & Hot-Reload ===
REPO_URL = "https://github.com/BoldHu/multiagent-speak.git"  # Edit once with your repo URL
REPO_DIR = Path("/content/when2speak") if IN_COLAB else Path(".").resolve()

import importlib, sys, types

if IN_COLAB:
    if not REPO_DIR.exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull --ff-only
    %cd {REPO_DIR}
    try:
        import typesafe_sdk, duckdb, typer, tenacity  # noqa: F401
        !pip install -q -e . --no-deps
    except ImportError:
        !pip install -q -e .

# Ensure `src/` is immediately importable in a fresh Colab kernel without restarting Python
_src_dir = str((REPO_DIR / "src").resolve())
if _src_dir not in sys.path:
    sys.path.insert(0, _src_dir)

# Colab Python 3.12+/3.13 removed stdlib `imp`, which older IPython autoreload still imports.
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp_shim = types.ModuleType("imp")
        _imp_shim.reload = importlib.reload
        sys.modules["imp"] = _imp_shim

try:
    _ip = get_ipython()
    if _ip is not None:
        _ip.run_line_magic("load_ext", "autoreload")
        _ip.run_line_magic("autoreload", "2")
except Exception as _exc:
    print(f"Skipping autoreload ({_exc}); restart runtime after git pull if needed.")


In [ ]:
# === CELL 2: Bootstrap Secrets & Resolved Model Config (Never Prints Raw Keys) ===
import json
from dial_skill.utils.secrets import describe_secret_sources
from dial_skill.llm.preflight import load_resolved_models, resolve_model_for_role
from dial_skill.config import load_models_config, load_dial_skill_config, load_experiments_config

print("Secret lookup diagnostics:", json.dumps(describe_secret_sources(drive_root=DRIVE_ROOT), indent=2))
MODELS_CFG = load_models_config()
DIAL_CFG = load_dial_skill_config()
EXP_CFG = load_experiments_config()
RESOLVED_MODELS = load_resolved_models(drive_root=DRIVE_ROOT)
print("Resolved models:", RESOLVED_MODELS.model_dump() if RESOLVED_MODELS else "Not yet generated (run Notebook 00)")


## Step 1: Configure Mode & Shared Cached Clients

In [ ]:
from dial_skill.benchmarks.pagemantle_env import PageMantleEnv
from dial_skill.benchmarks.when2speak import load_when2speak_split
from dial_skill.eval.runners import estimate_run_cost_usd
from dial_skill.llm.jev_client import TriageClient
from dial_skill.llm.openai_client import OpenAIClient
from dial_skill.schemas import PageMantleTask
from dial_skill.utils.cache import SQLiteAPICache
from dial_skill.utils.checkpoint import save_run_metadata

MODE = "pilot"  # Change to "full" (1,000 W2S turns + 32 pm_eval tasks) for paper Table 4 & Fig 4!
RUN_ABL = DRIVE_ROOT / "runs" / "ablations_a1_a7" / MODE
RUN_PARETO = DRIVE_ROOT / "runs" / "scaling_pareto" / MODE
print("Estimated Ablation Cost:", json.dumps(estimate_run_cost_usd(mode=MODE, include_ablations=True), indent=2))
save_run_metadata(RUN_ABL, {"mode": MODE})

cache = SQLiteAPICache(DRIVE_ROOT / "caches" / "api_cache.sqlite")
oai = OpenAIClient(models_cfg=MODELS_CFG, cache=cache)
luna_model = resolve_model_for_role("luna", MODELS_CFG, RESOLVED_MODELS)
terra_model = resolve_model_for_role("terra", MODELS_CFG, RESOLVED_MODELS)
backend = RESOLVED_MODELS.triage_backend if RESOLVED_MODELS else MODELS_CFG.roles.triage_backend
triage_client = TriageClient(models_cfg=MODELS_CFG, cache=cache, openai_client=oai, luna_model=luna_model, backend=backend)

# Load validation-calibrated (lambda, tau_dm) from Notebook 03 if available (Invariant #8)
w2s_cal = None
for cal_cand in [
    DRIVE_ROOT / "runs" / "main_when2speak" / "main" / "calibration.json",
    DRIVE_ROOT / "runs" / "main_when2speak" / "pilot" / "calibration.json",
]:
    if cal_cand.is_file():
        w2s_cal = json.loads(cal_cand.read_text(encoding="utf-8"))
        break
w2s_cal_luna = (w2s_cal.get("luna") if isinstance(w2s_cal, dict) and "luna" in w2s_cal else w2s_cal) or {
    "best_lambda": DIAL_CFG.gate.lambda_disruption,
    "best_tau_dm": DIAL_CFG.gate.tau_dm,
}
print("Using When2Speak validation calibration for ablations:", w2s_cal_luna)

w2s_split = "pilot" if MODE == "pilot" else "main"
w2s_turns = load_when2speak_split(w2s_split, local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
if MODE == "pilot":
    w2s_turns = w2s_turns[:200]
else:
    w2s_turns = w2s_turns[:1000]

pm_eval_path = DRIVE_ROOT / "datasets" / "pagemantle_ext" / "pm_eval_32.jsonl"
pm_eval_all = [PageMantleTask.model_validate_json(l) for l in pm_eval_path.read_text(encoding="utf-8").splitlines() if l.strip()]
if MODE == "pilot":
    pm_eval = [t for t in pm_eval_all if t.expected_action == "INTERVENE"][:4] + [
        t for t in pm_eval_all if t.expected_action == "SILENT"
    ][:4]
else:
    pm_eval = pm_eval_all
env = PageMantleEnv(dataset_dir=DRIVE_ROOT / "datasets" / "pagemantle")
env.ensure_extracted()
print(f"Ablation sets -> W2S: {len(w2s_turns)} turns | PageMantle: {len(pm_eval)} tasks (traps={sum(1 for t in pm_eval if t.expected_action=='SILENT')})")


## Step 2: Run 7-Point Ablation Matrix (`A1`–`A7`)

In [ ]:
from dial_skill.core.pipeline import DialSkillAgent
from dial_skill.core.skill_store import SkillStore
from dial_skill.eval.judge import BlindedDualJudge
from dial_skill.eval.metrics import compute_pagemantle_metrics, compute_when2speak_metrics
from dial_skill.eval.runners import run_pagemantle_eval, run_when2speak_eval

judge = BlindedDualJudge(openai_client=oai, primary_judge_model=terra_model)
y_true = [t.label_speak for t in w2s_turns]
ablation_results = {}

variants = [{"id": "Full_DIAL_Skill", "overrides": {}}] + EXP_CFG["ablations"]
for spec in variants:
    vid = spec["id"]
    pm_cfg = load_dial_skill_config(overrides=spec.get("overrides", {}))
    w2s_overrides = dict(spec.get("overrides", {}))
    # Apply validation-calibrated (lambda, tau_dm) to dual/combined When2Speak gates
    gate_mode = pm_cfg.ablation.gate_mode
    if gate_mode in ("dual_margin", "single_combined"):
        w2s_overrides = {
            **w2s_overrides,
            "gate": {
                "lambda_disruption": w2s_cal_luna["best_lambda"],
                "tau_dm": w2s_cal_luna["best_tau_dm"],
            },
        }
    w2s_cfg = load_dial_skill_config(overrides=w2s_overrides)

    w2s_agent = DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=luna_model,
        skill_store=SkillStore.from_json_file(), cfg=w2s_cfg, method_name=vid,
    )
    pm_agent = DialSkillAgent(
        triage_client=triage_client, openai_client=oai, gate_model=luna_model, executor_model=terra_model,
        skill_store=SkillStore.from_json_file(), cfg=pm_cfg, method_name=vid,
    )
    w2s_eps = await run_when2speak_eval(
        agent=w2s_agent, turns=w2s_turns, output_jsonl=RUN_ABL / f"w2s_{vid}.jsonl", resume=True, generate_text=False
    )
    pm_eps = await run_pagemantle_eval(
        agent=pm_agent, tasks=pm_eval, env=env, output_jsonl=RUN_ABL / f"pm_{vid}.jsonl", resume=True
    )
    pm_scores = [await judge.grade_pagemantle_episode(t, e) for t, e in zip(pm_eval, pm_eps)]
    ablation_results[vid] = {
        "when2speak": compute_when2speak_metrics(y_true, w2s_eps, n_bootstrap=300),
        "pagemantle": compute_pagemantle_metrics(pm_eval, pm_eps, pm_scores, n_bootstrap=300),
    }
    print(
        f"[{vid:28s}] W2S F1={ablation_results[vid]['when2speak']['macro_f1']:.3f} "
        f"(FIR={ablation_results[vid]['when2speak']['fir']:.3f}) | "
        f"PM Overall={ablation_results[vid]['pagemantle']['overall_score']:.2f} "
        f"(Aud={ablation_results[vid]['pagemantle']['audience_and_location']:.2f})"
    )

(RUN_ABL / "ablation_summary.json").write_text(json.dumps(ablation_results, indent=2), encoding="utf-8")
print("Cache stats after ablation sweep:", cache.stats())


## Step 3: Zero-Cost Continuous $\lambda$-Pareto FIR–MIR Sweep

In [ ]:
from dial_skill.eval.runners import sweep_lambda_pareto_from_episodes
from dial_skill.schemas import EpisodeResult
from dial_skill.utils.checkpoint import load_jsonl

# Prefer full When2Speak-Main predictions if available for high-resolution Pareto curve
main_preds = DRIVE_ROOT / "runs" / "main_when2speak" / "main" / "preds_DIAL_Skill_luna.jsonl"
if MODE != "pilot" and main_preds.is_file():
    pareto_turns = load_when2speak_split("main", local_dir=DRIVE_ROOT / "datasets" / "when2speak", seed=EXP_CFG["seed"])
    full_w2s_eps = [EpisodeResult.model_validate(r) for r in load_jsonl(main_preds)]
else:
    pareto_turns = w2s_turns
    full_w2s_eps = [EpisodeResult.model_validate(r) for r in load_jsonl(RUN_ABL / "w2s_Full_DIAL_Skill.jsonl")]

pareto_rows = sweep_lambda_pareto_from_episodes(
    turns=pareto_turns,
    episodes=full_w2s_eps,
    lambdas=[0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0],
    tau_dm=float(w2s_cal_luna["best_tau_dm"]),
    theta_floor=DIAL_CFG.gate.theta_floor,
)
RUN_PARETO.mkdir(parents=True, exist_ok=True)
(RUN_PARETO / "lambda_pareto.json").write_text(json.dumps(pareto_rows, indent=2), encoding="utf-8")
print("FIR-MIR Pareto Frontier:", json.dumps(pareto_rows, indent=2))
